In [ ]:
%env CUDA_VISIBLE_DEVICES=0

import hashlib

from urllib import request
from pathlib import Path

import numpy as onp

import jax.numpy as jnp
import jax

import optax

import matplotlib.pyplot as plt

import flax

from chemtrain import quantity, trainers, ensemble
from chemtrain.data import preprocessing

from jax_md import space, partition, simulate
from jax_md_mod import custom_quantity, custom_energy

# Coarse-Grained Water MLPs

In this tutorial, you will build two coarse-grained water machine-learning
potentials (MLPs), implemented here with multilayer perceptrons. The first model
is purely radial, while the second uses a learned local descriptor and Cartesian
tensor operations to represent many-body interactions.

You will then train the models in the framework **chemtrain**.
First, you will train both models on microscopic reference data with force
matching and compare them experimental structural measurements. Then, you will then refine the many-body model against these experimental measurements using the DiffTRe algorithm.

## Data Loading

The following cell loads reference mapped position and force data. For this tutorial, we express the coordinates in fractional units with respect to the simulation box.

In [ ]:
positions_url = "https://drive.usercontent.google.com/download?id=1wVJ3cEakl5IngvuZycp0wEOsn7EQyBD8&export=download&authuser=0&confirm=t&uuid=916ca889-af2d-458f-8e97-5b1c23ea39ba&at=ALBwUgm15BEzvzbLeMs6UUzr45Lb:1777900857190"
forces_url = "https://drive.usercontent.google.com/download?id=1EgSrBa5e2-dlgG0X8fIDATbR9q3xSFwv&export=download&authuser=0&confirm=t&uuid=994f428e-a771-427f-b769-fce10ec66dd1&at=ALBwUglSh_WZNFUOOdddmeZC3JTT:1777900862535"

box = 3.12867066 # DO NOT CHANGE THIS VALUE
box = jnp.array([box, box, box])

data_dir = Path("data")
data_dir.mkdir(exist_ok=True)
positions_path = data_dir / "positions.npy"
forces_path = data_dir / "forces.npy"

if not positions_path.exists():
    request.urlretrieve(positions_url, positions_path)
if not forces_path.exists():
    request.urlretrieve(forces_url, forces_path)
 
assert hashlib.sha256(forces_path.read_bytes()).hexdigest() == 'b724eb35cc1890f5866decb0ac9b04a3f4a624bf9a323ceb89d2e99314a43687'
assert hashlib.sha256(positions_path.read_bytes()).hexdigest() == '873c7ee08bd6238531ec10dbc41e2f9651d128cdc6c61dd7dfa29f22e931bdbc'

force_dataset = preprocessing.get_dataset(forces_path, subsampling=10)
position_dataset = preprocessing.get_dataset(positions_path, subsampling=10)

position_dataset = preprocessing.scale_dataset_fractional(
    position_dataset, box
)

dataset = {
    "R": position_dataset,
    "F": force_dataset
}

print("Loaded dataset:\n\tpositions shape:", position_dataset.shape, "\n\tforces shape:", force_dataset.shape)

### Radial MLP

We start with a simple learned **pairwise** MLP. For every neighbor pair $(i,j)$,
the network predicts a feature, for example a pairwise energy contribution, from
the inter-particle distance and the species of the two particles. Because the
input depends only on the scalar distance $r_{ij}$, this architecture is
**radial** and does not encode any angular or many-body information.

For each neighbor pair we form the displacement vector $\mathbf{r}_{ij}$ and its length
$$
r_{ij} = \|\mathbf{r}_{ij}\|.
$$
We then expand the normalized distance $r_{ij}/r_\mathrm{cut}$ in a Gaussian radial basis,
$$
\phi_n(r_{ij}) =
\exp\left[
-\frac{1}{2}
\left(
\frac{r_{ij}/r_\mathrm{cut} - \mu_n}{\sigma_n}
\right)^2
\right],
\qquad n=1,\dots,N_\mathrm{basis},
$$
where the centers $\mu_n$ are equally spaced in $[0,1]$.

This radial basis is concatenated with learned embeddings of the sender and receiver species,
$$
\mathbf{x}_{ij}^{(0)}
=
\left[
\mathbf{e}(Z_i),\,
\mathbf{e}(Z_j),\,
\phi(r_{ij})
\right].
$$
The resulting edge feature is passed through a multilayer perceptron,
$$
\mathbf{x}_{ij}^{(\ell+1)}
=
\mathrm{silu}\!\left(
W^{(\ell)} \mathbf{x}_{ij}^{(\ell)} + \mathbf{b}^{(\ell)}
\right),
$$
and mapped to the final output by a linear readout,
$$
\mathbf{y}_{ij}
=
f_\mathrm{env}(r_{ij})
\, W_\mathrm{out}\mathbf{x}_{ij}^{(L)}.
$$

The polynomial envelope
$$
f_\mathrm{env}(r)
=
1
-
\frac{(p+1)(p+2)}{2}
\left(\frac{r}{r_\mathrm{cut}}\right)^p
+
p(p+2)
\left(\frac{r}{r_\mathrm{cut}}\right)^{p+1}
-
\frac{p(p+1)}{2}
\left(\frac{r}{r_\mathrm{cut}}\right)^{p+2}
$$
ensures that the predicted interaction smoothly goes to zero at the cutoff.

In the code above, the neighbor vectors `vectors`, the inter-particle distances `lengths`, the Gaussian radial basis, the species embeddings, and the envelope function are already provided. The concatenated input features `x` are therefore already constructed.

What is still missing is the actual forward pass through the fully connected
network: apply each dense layer to `x`, use the `silu` activation after every
hidden layer, then compute the final readout `y` and multiply it by the
envelope so that the output vanishes smoothly at the cutoff.


In [ ]:
class GaussianRadialBasis:

    def __init__(self, num_basis: int = 8, cutoff: float = 0.5, envelope_p: int = 5):
        super().__init__()
        
        self.num_basis = num_basis
        self.cutoff = cutoff
        self.envelope_p = envelope_p


    def __call__(self, r):
        r /= self.cutoff # normalize distances to the cutoff

        # Use equally spaced Gaussians in [0, cutoff]
        centers = jnp.linspace(0, 1, self.num_basis)
        widths = jnp.full_like(centers, 0.5 * (centers[1] - centers[0]))

        # Compute Gaussian basis functions
        gaussians = jnp.exp(-0.5 * ((r[..., None] - centers) / widths) ** 2)
        # Apply envelope function to ensure smooth cutoff

        # Returns shape (num_edges, num_basis)
        return gaussians


class PolynomialEnvelope:
    
    def __init__(self, p: int = 5, cutoff: float = 0.5):
        self.p = p
        self.cutoff = cutoff

    def __call__(self, r):
        r /= self.cutoff
        return (
            1 - ((self.p + 1) * (self.p + 2) / 2) * r**self.p
            + self.p * (self.p + 2) * r**(self.p + 1)
            - (self.p * (self.p + 1) / 2) * r**(self.p + 2)
        )[..., None]


class RadialMLP(flax.linen.Module):

    num_layers: int = 2
    hidden_size: int = 64
    output_size: int = 1
    num_basis: int = 8
    num_species: int = 5
    r_cutoff: float = 0.5

    def setup(self):
        self.embedd = flax.linen.Embed(
            num_embeddings=self.num_species, features=self.hidden_size // 2)
        self.rbf = GaussianRadialBasis(
            num_basis=self.num_basis, cutoff=self.r_cutoff)
        self.envelope = PolynomialEnvelope(p=6, cutoff=self.r_cutoff)

        self.layers = [
            flax.linen.Dense(self.hidden_size) for _ in range(self.num_layers)
        ]

        self.readout = flax.linen.Dense(
            self.output_size, use_bias=False)



    def __call__(self, vectors, species, senders, receivers):
        lengths = jnp.linalg.norm(vectors, axis=-1)            
        
        x = jnp.concat([
            self.embedd(species)[senders],
            self.embedd(species)[receivers],
            self.rbf(lengths)
            ], axis=-1
        )

        #######################################################################
        # FCNN from distance and species features to radial features
        #######################################################################

        # TODO: Implement
        #
        # Inputs:
        #  - x: shape (num_edges, hidden_size) inputs to the FCNN
        # Outputs:
        #  - y: shape (num_edges, output_size) outputs of the readout layer

        y = None

        #######################################################################

        return y

### Building the graph input

Before applying the network, we must convert each configuration into a neighbor graph using a cutoff radius $r_\mathrm{cut}$. The neighbor graph defines edges
$$
(i,j)\in\mathcal E \quad \text{if} \quad \|\mathbf r_{ij}\| < r_\mathrm{cut},
$$
where $\mathbf r_{ij}$ is computed with periodic boundary conditions.

The function `allocate_neighborlist(...)` estimates how many neighbors and edges are needed for the dataset and allocates a dense neighbor list of fixed size. The helper `vectors_from_nbrs(...)` then turns this neighbor list into a graph, represented by arrays:
- `senders`: source particle indices,
- `receivers`: neighbor indices,
- `vectors`: displacement vectors $\mathbf r_{ij}$ for all edges.

Since JAX requires static shapes under `jit`, invalid or padded neighbor entries are replaced by a fixed padding vector, and the total number of edges is truncated to at most `max_edges`. This gives a stable graph representation that can be processed efficiently by the model.

Finally, `initialize_model(...)` returns two compiled functions: `init_fn` initializes the model parameters, and `apply_fn` evaluates the model on a configuration. In both cases, the neighbor list is first converted into edge features and then passed to the neural network.


In [ ]:
r_cutoff = 0.5

def initialize_model(architecture, config, r_cutoff, displacement_fn, max_edges):
    model = architecture(**config, r_cutoff=r_cutoff)

    def vectors_from_nbrs(neighbor, position):
        num_particles, max_nbrs = neighbor.idx.shape
        senders = jnp.repeat(jnp.arange(num_particles), max_nbrs)
        receivers = neighbor.idx.flatten()

        mask = receivers < num_particles
        _, select = jax.lax.top_k(mask, k=onp.min([max_edges, mask.size]))
        receivers = jnp.take_along_axis(receivers, select, axis=0)
        senders = jnp.take_along_axis(senders, select, axis=0)

        padding = jnp.ones((3,)) * r_cutoff / jnp.sqrt(3)
        vectors = jax.vmap(displacement_fn)(position[senders], position[receivers])

        vectors = jnp.where(senders[:, None] < num_particles, vectors, padding[None, :])
        vectors = jnp.where(receivers[:, None] < num_particles, vectors, padding[None, :])

        return vectors, senders, receivers

    @jax.jit
    def init_fn(rng, position, species, neighbor):
        vectors, senders, receivers = vectors_from_nbrs(neighbor, position)

        return model.init(rng, vectors, species, senders, receivers)

    @jax.jit
    def apply_fn(params, position, species, neighbor):
        vectors, senders, receivers = vectors_from_nbrs(neighbor, position)

        return model.apply(params, vectors, species, senders, receivers)

    return init_fn, apply_fn

displacement_fn, shift_fn = space.periodic_general(box, fractional_coordinates=True)

nbrs_init, (_, max_edges, *_) = preprocessing.allocate_neighborlist(
    dataset, displacement_fn, box, r_cutoff,
    capacity_multiplier=2.5, batch_size=10,
    format=partition.Dense)

max_edges *= 2

## Actually initizalize the model

init_fn, apply_fn = initialize_model(
    RadialMLP,
    {
        "num_layers": 2,
        "hidden_size": 64,
        "num_basis": 8,
        "num_species": 1
    }, r_cutoff, displacement_fn, max_edges
)

r_init = dataset["R"][0]
nbrs_init = nbrs_init.update(r_init)
species_init = jnp.zeros(r_init.shape[0], dtype=int)

key = jax.random.PRNGKey(0)
key, split = jax.random.split(key)
init_params = init_fn(split, r_init, species_init, nbrs_init)

### Trainable energy model

To obtain a stable energy model, we combine the learned network with a simple **Lennard-Jones prior**. The total energy is
$$
U(\mathbf R) = U_{\mathrm{prior}}(\mathbf R) + U_{\theta}(\mathbf R).
$$

Here, `prior_energy_fn` defines a neighbor-list Lennard-Jones potential with parameters `epsilon=0.5` and `sigma=0.31`.
The neural network then learns an additional correction on top of this prior.

The function `radial_energy_fn_template(params)` returns an energy function with fixed model parameters `params`. For a given configuration, it first assigns the particle species. Since the system only contains water, all particles are mapped to the same species index. It then evaluates the prior energy and adds the sum of all learned edge contributions,
$$
U_{\theta}(\mathbf R) = \sum_{(i,j)\in\mathcal E} y_{ij}.
$$

In this way, the prior provides a physically reasonable baseline, while the neural network learns the remaining part of the interaction from the data.

The general structure of the trainable energy model is already given. You
still have to implement the evaluation of the prior and MLP and compute the
total potential.

In [ ]:
prior_energy_fn = custom_energy.customn_lennard_jones_neighbor_list(
    displacement_fn, box, epsilon=0.5, sigma=0.31,
    r_onset=0.9*r_cutoff, r_cutoff=r_cutoff,
    initialize_neighbor_list=False
)


def radial_energy_fn_template(params):

    def energy_fn(position, neighbor, **kwargs):
        # We only have water
        species = jnp.zeros((position.shape[0],), dtype=int)

        #######################################################################
        # Hybrid delta-ML potential energy function
        #######################################################################
        
        # TODO: Implement
        #
        # Inputs:
        #  - position: (n, 3) array of particle positions
        #  - species: (n,) array of particle species
        #  - neighbor: (Any) neighbor list for the current configuration
        #  - params: (Any) trained radial-network parameters from the outer scope
        #
        # Outputs:
        #  - pot: total energy of the configuration

        pot = None

        #######################################################################

        return pot

    return energy_fn

### Force Matching Optimization

We now train the radial energy model by **force matching**. Given an energy function
$$
U_\theta(\mathbf R),
$$
the predicted forces are obtained as
$$
\mathbf F_\theta(\mathbf R) = - \nabla_{\mathbf R} U_\theta(\mathbf R).
$$
The training objective is to minimize the discrepancy between predicted and reference forces over the dataset,
$$
\mathcal L_{\mathrm{FM}}(\theta)
=
\frac{1}{|\mathcal D|}\sum_{(\mathbf R,\mathbf F^{\mathrm{ref}})\in\mathcal D}
\frac{1}{3N}
\left\|
\mathbf F_\theta(\mathbf R)-\mathbf F^{\mathrm{ref}}
\right\|_2^2.
$$

To optimize this objective, we choose a decreasing learning-rate schedule and use Adam as the optimizer. The `ForceMatching` trainer combines the initial parameters, the optimizer, the energy model, and the preallocated neighbor list into a training loop that repeatedly evaluates the force-matching loss and updates the parameters.

Finally, `set_datasets(...)` splits the data into training and validation sets. The training set is used to minimize $\mathcal L_{\mathrm{FM}}$, while the validation set provides an estimate of the same error on unseen configurations. This makes it possible to monitor both optimization progress and generalization during training.


In [ ]:
# Define optimizer
num_iter = 100
lr_decay = optax.polynomial_schedule(init_value=1e-2, end_value=1e-3, power=2, transition_steps=num_iter)
optimizer = optax.adam(learning_rate=lr_decay)

# Initialize trainer 
radial_fm_trainer = trainers.ForceMatching(
    init_params=init_params,
    optimizer=optimizer,
    energy_fn_template=radial_energy_fn_template,
    nbrs_init=nbrs_init,
    batch_per_device=32
)

# Specify reference data
radial_fm_trainer.set_datasets(dataset, train_ratio=0.7, val_ratio=0.2)

Calling `radial_fm_trainer.train(num_iter)` starts the actual optimization and runs the force-matching training for ``num_iter`` steps. In each step, the trainer samples batches from the training set, evaluates the model forces, compares them to the reference forces, and updates the model parameters using the optimizer defined above.

During training, the validation set is used to monitor performance on unseen data. This makes it possible to check whether the model is improving and whether it generalizes beyond the training configurations.

In [ ]:
radial_fm_trainer.train(num_iter)

The next cell visualizes the force-matching loss during training.
Both the training and validation losses decrease, but they remain relatively
large. This is typical in coarse-graining: the reduced model cannot reproduce
the underlying atomistic forces exactly, and the target forces already represent
an effective many-body projection onto the coarse-grained variables.

In [ ]:
plt.plot(radial_fm_trainer.train_losses, label="Train Loss")
plt.plot(radial_fm_trainer.val_losses, label="Val Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.show()

### Evaluating the learned pair potential

This snippet probes the learned energy model on a simple two-particle system. We construct 100 configurations in which one particle is fixed at the origin and the second particle is placed at distances between `0.25` and `0.6`. The displacement is chosen along the diagonal direction, but since the radial model only depends on the distance, the exact direction is not important.

A neighbor list is then initialized for this test system. The helper function `single_eval(...)` updates the neighbor list for a given configuration and evaluates the energy using the trained model parameters. Since the positions in the dataset are represented in fractional coordinates, the particle positions are divided by `box` before the energy is computed.

Finally, the list comprehension evaluates the energy for all test distances
using `radial_fm_trainer.best_params`. The resulting array `res` lets us inspect
the effective pair interaction learned by the radial force-matching model.

In [ ]:
r_test = jnp.ones((3,))[None, :] * jnp.linspace(0.25, 0.6, 100)[:, None] / jnp.sqrt(3)
r_test = r_test[:, None, :] * jnp.asarray([1, 0])[None, :, None]

nbrs_test = partition.neighbor_list(
    displacement_fn, box, r_cutoff=r_cutoff, format=partition.Dense
)
nbrs_test = nbrs_test.allocate(r_test[0] / box)

@jax.jit
def single_eval(params, r):
    r /= box
    nbrs = nbrs_test.update(r)
    return radial_energy_fn_template(params)(r, nbrs)

res = [single_eval(radial_fm_trainer.best_params, r) for r in r_test]

This plot shows the effective pair potential learned by the radial model as a
function of the distance between two particles. The x-axis contains the
inter-particle distance from the test configurations, and the y-axis shows the
corresponding predicted energy.

Visualizing the energy in this way makes it easy to check whether force
matching has produced a physically plausible isotropic interaction: for example,
a short-range repulsion, a favorable intermediate distance, and a smooth decay
toward zero near the cutoff.

In [ ]:
plt.plot(onp.linalg.norm(r_test[:, 0, :] - r_test[:, 1, :], axis=-1), res)
plt.xlabel("Distance")
plt.ylabel("Energy")
plt.title("Energy vs. Distance")
plt.show()


### Comparing to structural reference data

So far, we have trained the model to reproduce forces and inspected the resulting
pair interaction. We now move to a structural validation and compare the model
against reference RDF and ADF data.

The radial distribution function (RDF) measures distance correlations, whereas
the angular distribution function (ADF) probes local orientational order. These
observables therefore test whether the learned coarse-grained interaction
reproduces the liquid structure for the right physical reasons.

We will load the reference curves, define matching observables for the
simulation, and then compare trajectory-averaged predictions to the targets.

In [ ]:
rdf_url = "https://raw.githubusercontent.com/tummfm/difftre/92c0790b89f0d570ed9f79663e6c06580f598345/data/experimental/O_O_RDF.csv"
adf_url = "https://raw.githubusercontent.com/tummfm/difftre/92c0790b89f0d570ed9f79663e6c06580f598345/data/experimental/O_O_O_ADF.csv"

if not (data_dir / "O_O_RDF.csv").exists():
    request.urlretrieve(rdf_url, data_dir / "O_O_RDF.csv")

if not (data_dir / "O_O_O_ADF.csv").exists():
    request.urlretrieve(adf_url, data_dir / "O_O_O_ADF.csv")

assert hashlib.sha256((data_dir / "O_O_RDF.csv").read_bytes()).hexdigest() == 'b9701ad4dd35b47985a5f4e4cd01b8351ae384a99282ad8f91dfe9e5be555192'
assert hashlib.sha256((data_dir / "O_O_O_ADF.csv").read_bytes()).hexdigest() == 'a737a25056524f04b461d899726289bfe77a4ca1308dad6151a053e0d8651613'

r_rdf, rdf = onp.loadtxt(data_dir / "O_O_RDF.csv", unpack=True)
alpha_adf, adf = onp.loadtxt(data_dir / "O_O_O_ADF.csv", unpack=True) 

The following cell discretizes the experimental RDF and ADF and defines the
corresponding observables for the simulation.
A single simulated configuration gives only an instantaneous structural
snapshot, whereas the experimental quantities are ensemble averages. For ergodic systems, we can use MD simulations to compute these averages via
$$
\langle a \rangle_\theta = \lim_{T\to\infty}\frac{1}{T}\int_0^T a(\mathbf{r}(t,\theta))\,dt \approx \frac{1}{S}\sum_{s=1}^S a(\mathbf{r}(t_s, \theta)),
$$
which we approximate practically by a finite length discretized trajectory of $S$ samples.

By placing the RDF and ADF on fixed distance and angle grids, we can accumulate
these structural correlations along the trajectory and compare the resulting
averaged curves directly to the experimental targets.

In [ ]:
rdf_discretization = custom_quantity.rdf_discretization(0.9, nbins=300)
adf_disc = custom_quantity.adf_discretization(nbins=150)

rdf_params = custom_quantity.RDFParams(
    jnp.interp(rdf_discretization[0], r_rdf, rdf), *rdf_discretization
)
adf_params = custom_quantity.ADFParams(
    jnp.interp(adf_disc[0], alpha_adf, adf), *adf_disc, 0.318, 0.0
)

rdf_fn = custom_quantity.init_rdf(displacement_fn, rdf_params)
adf_fn = custom_quantity.init_adf_nbrs(displacement_fn, adf_params, r_init=r_init, nbrs_init=nbrs_init)

quantities = {
    "rdf": rdf_fn,
    "adf": adf_fn
}

rdf_observable = quantity.observables.init_traj_mean_fn("rdf")
adf_observable = quantity.observables.init_traj_mean_fn("adf")

This cell sets up a simulation of the trained radial model. We define the
Langevin parameters and sampling schedule, assign a mass to each coarse-grained
water particle, and choose a random configuration from the dataset as the
initial structure.

Using this configuration, we update the neighbor list, initialize the
simulator, and construct the corresponding `reference_state` with the best
radial force-matching parameters. This state serves as the starting point for
the trajectory generation in the next step.

In [ ]:
dt = 0.01
gamma = 100.
kT = quantity.kb * 300.0

timings = ensemble.sampling.process_printouts(
    time_step=dt, total_time=60.0, print_every=0.1,
    t_equilib=10.0
)

masses = jnp.ones((position_dataset.shape[1],)) * 18.02

key, split = jax.random.split(key)
selection = jax.random.choice(
    split, jnp.arange(position_dataset.shape[0]), shape=(1,), replace=False)
r_init = position_dataset[selection, ...].squeeze(0)
nbrs_init = nbrs_init.update(r_init)

init_ref_state, sim_template = ensemble.sampling.initialize_simulator_template(
    simulate.nvt_langevin, shift_fn=shift_fn, nbrs=nbrs_init,
    init_with_PRNGKey=True, extra_simulator_kwargs={"kT": kT, "gamma": gamma, "dt": dt}
)

key, split = jax.random.split(key)
reference_state = init_ref_state(
    split, r_init,
    energy_or_force_fn=radial_energy_fn_template(radial_fm_trainer.best_params),
    init_sim_kwargs={"mass": masses, "neighbor": nbrs_init}
)


We now combine the simulator, learned energy model, sampling protocol, and
structural observables into a trajectory generator. This lets us simulate from
the previously initialized reference state with the current model parameters and
record the quantities of interest along the trajectory.

In [ ]:
trajectory_generator = ensemble.sampling.trajectory_generator_init(
    sim_template, radial_energy_fn_template, timings, quantities)

trajectory = trajectory_generator(radial_fm_trainer.params, reference_state, box=box)

print(f"Average kT: {trajectory.aux['kT'].mean()}")

The figure compares the trajectory-averaged structural observables of the
radial model to the corresponding experimental reference curves: the left panel
shows the RDF, and the right panel shows the ADF.

The RDF is reproduced only qualitatively, with visible deviations in peak
heights and positions. This means that the effective pair structure is captured
only approximately.

The discrepancy is substantially larger for the ADF, which probes three-body
angular correlations and local orientational order. This is the key limitation
of a purely radial coarse-grained model: isotropic pair interactions can recover
part of the distance-dependent structure, but they do not explicitly represent
the directional many-body correlations that organize liquid water.

In [ ]:
plt.figure(figsize=(12, 5), layout="constrained")
plt.subplot(1, 2, 1)
plt.plot(rdf_params.rdf_bin_centers, rdf_params.reference, label="Experimental RDF")
plt.plot(rdf_params.rdf_bin_centers, rdf_observable(trajectory.aux), label="Simulated RDF")
plt.xlabel("Distance")
plt.ylabel("RDF")
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(adf_params.adf_bin_centers, adf_params.reference, label="Experimental ADF")
plt.plot(adf_params.adf_bin_centers, adf_observable(trajectory.aux), label="Simulated ADF")
plt.legend()
plt.xlabel("Angle")
plt.ylabel("ADF")

### Local many-body MLP

We now move from a radial pair model to a local many-body MLP. Instead of assigning features only to edges, we first build a fixed-size descriptor for each central particle from its local environment
$$
\chi_i = \{(r_{ij}, Z_j) \mid j \in \mathcal N(i)\},
$$
and then map this descriptor to a particle-wise output.

As before, the radial network processes each neighbor pair and returns learned edge features. These are split into scalar and directional channels,
$$
(s_{ij}, v_{ij}) = \mathrm{RadialMLP}(\mathbf r_{ij}, Z_i, Z_j).
$$
The scalar channels are summed over the neighborhood,
$$
S_i = \sum_{j\in\mathcal N(i)} s_{ij},
$$
while the directional channels are multiplied by the unit vectors
$$
\hat{\mathbf r}_{ij} = \frac{\mathbf r_{ij}}{\|\mathbf r_{ij}\|},
$$
and pooled as
$$
\mathbf V_i = \sum_{j\in\mathcal N(i)} v_{ij}\hat{\mathbf r}_{ij}.
$$

The quantity $\|\mathbf V_i\|^2$ is rotationally invariant, but still depends on relative neighbor directions. After expansion, it contains scalar products of the form
$$
\hat{\mathbf r}_{ij}\cdot\hat{\mathbf r}_{ik} = \cos \angle(jik),
$$
so it carries angular information.

Finally, we combine scalar and directional information into the local descriptor
$$
\mathbf G_i = [S_i, \|\mathbf V_i\|^2],
$$
and pass it through a second MLP and a linear readout. In this way, the model
remains local and permutation-invariant while going beyond purely radial
interactions.

In the next cell, the radial edge model `self.radial`, the hidden layers, and
the readout are already provided. What remains is the forward construction of
the local many-body descriptor: compute edge features with the radial model,
split them into scalar and directional channels, convert the directional part
into vectors using the normalized displacement directions, pool both
contributions over neighbors with `segment_sum`, and build the invariant node
features by concatenating the summed scalar channels with the squared norm of
the pooled vector features. These node features are then passed through the
final MLP and readout.


In [ ]:
class LocalManybodyMLP(flax.linen.Module):

    num_layers: int = 2
    hidden_size: int = 64
    num_species: int = 5
    r_cutoff: float = 0.5
    num_basis: int = 8
    readout_size: int = 1

    def setup(self):
        self.radial = RadialMLP(
            num_layers=self.num_layers,
            hidden_size=self.hidden_size,
            num_basis=self.num_basis,
            r_cutoff=self.r_cutoff,
            output_size=self.hidden_size
        )
        self.layers = [
            flax.linen.Dense(self.hidden_size) for _ in range(self.num_layers)
        ]
        self.readout = flax.linen.Dense(self.readout_size, use_bias=False)

    def __call__(self, vectors, species, senders, receivers):
        # Pool over all neighbors
        s, v = jnp.split(self.radial(vectors, species, senders, receivers), 2, axis=-1)

        #######################################################################
        # Neighbor pooling and per-particle feature construction
        #######################################################################

        # TODO: Implement
        #
        # Inputs:
        #  - s: shape (num_edges, hidden_size // 2) scalar features
        #  - v: shape (num_edges, hidden_size // 2) vector features
        # Outputs:
        #  - node_feats: shape (num_particles, hidden_size) per-particle features

        node_feats = None

        #######################################################################

        for layer in self.layers:
            node_feats = layer(node_feats)
            node_feats = jax.nn.silu(node_feats)

        return self.readout(node_feats)

In [ ]:
architecture = LocalManybodyMLP
config = {
    "num_layers": 2,
    "hidden_size": 32,
    "num_basis": 8,
    "num_species": 1
}

###############################################################################
# Initializes the model with the same utilities as before
###############################################################################

# TODO: Implement
#
# Inputs:
# - architecture: the model architecture class (e.g. LocalManybodyMLP)
# - config: a dictionary of configuration parameters for the model architecture
# - r_cutoff, displacement_fn, max_edges: same as before
# - displacement_fn: the displacement function for computing neighbor vectors
#
# Outputs:
# - init_fn: a function that initializes the model parameters given a PRNGKey and input
# - apply_fn: a function that applies the model given parameters and input

init_fn = None
apply_fn = None

###############################################################################

key = jax.random.PRNGKey(0)
key, split = jax.random.split(key)
init_params = init_fn(split, r_init, species_init, nbrs_init)

print("Model initialized with parameters:")
print(jax.tree.map(lambda x: x.shape, init_params))

The next cell turns the local many-body network into a trainable energy model.
As in the radial case, we add a simple prior interaction so that the learned
model does not have to represent the entire short-range physics from scratch.
The neural network then learns a flexible correction on top of this baseline.

In [ ]:
prior_energy_fn = custom_energy.customn_lennard_jones_neighbor_list(
    displacement_fn, box, epsilon=0.5, sigma=0.31,
    r_onset=0.9*r_cutoff, r_cutoff=r_cutoff,
    initialize_neighbor_list=False
)


def manybody_energy_fn_template(params):

    def energy_fn(position, neighbor, **kwargs):

        # We only have CG water
        species = jnp.zeros((position.shape[0],), dtype=int)
        #######################################################################
        # Hybrid delta-ML potential energy function
        #######################################################################
        
        # TODO: Implement
        #
        # Inputs:
        #  - position: (n, 3) array of particle positions
        #  - species: (n,) array of particle species
        #  - neighbor: (Any) neighbor list for the current configuration
        #  - params: (Any) trained radial-network parameters from the outer scope
        #
        # Outputs:
        #  - pot: total energy of the configuration

        pot = None

        #######################################################################

        return pot

    return energy_fn

We first train the many-body model with force matching, exactly as before.
The main difference is representational power: because the descriptor
$\mathbf G_i$ contains angular information from the local environment, the
network can fit effective coarse-grained forces that a purely radial model
cannot express.

In [ ]:
# Define optimizer
num_iter = 100
lr_decay = optax.polynomial_schedule(init_value=1e-2, end_value=1e-3, power=2, transition_steps=num_iter)
optimizer = optax.adam(learning_rate=lr_decay)

# Initialize trainer 
manybody_fm_trainer = trainers.ForceMatching(
    init_params=init_params,
    optimizer=optimizer,
    energy_fn_template=manybody_energy_fn_template,
    nbrs_init=nbrs_init,
    batch_per_device=32
)

# Specify reference data
manybody_fm_trainer.set_datasets(dataset, train_ratio=0.7, val_ratio=0.2)

# Train the model
manybody_fm_trainer.train(num_iter)

The next cell compares the training and validation losses of the radial and
local many-body force-matching models.

The many-body MLP reaches lower training and validation errors than the radial
model: once $\chi_i$ is mapped to
a richer descriptor $\mathbf G_i$, the model can represent angular correlations
that are invisible to an isotropic pair potential. At the same time, the
absolute errors remain comparatively large. This is again typical of
coarse-grained force matching, where the reduced model cannot reproduce the
atomistic forces exactly.

In [ ]:
plt.plot(manybody_fm_trainer.train_losses, label="Many-Body Train Loss")
plt.plot(radial_fm_trainer.train_losses, label="Radial FM Train Loss")
plt.plot(manybody_fm_trainer.val_losses, label="Many-Body Val Loss")
plt.plot(radial_fm_trainer.val_losses, label="Radial FM Val Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.show()

We now simulate the best many-body force-matching model and compare its
trajectory-averaged RDF and ADF to the reference curves. This is the structural
validation step before DiffTRe.

Compared with the radial model, the key question is whether the additional
angular information in $\mathbf G_i$ improves not only the force-matching loss
but also the liquid structure, especially the ADF.

In [ ]:
key, split = jax.random.split(key)
reference_state = init_ref_state(
    split, r_init,
    energy_or_force_fn=manybody_energy_fn_template(manybody_fm_trainer.best_params),
    init_sim_kwargs={"mass": masses, "neighbor": nbrs_init}
)

trajectory_generator = ensemble.sampling.trajectory_generator_init(
    sim_template, manybody_energy_fn_template, timings, quantities)

trajectory = trajectory_generator(manybody_fm_trainer.best_params, reference_state, box=box)

print(f"Average kT: {trajectory.aux['kT'].mean()}")

In [ ]:
plt.figure(figsize=(12, 5), layout="constrained")
plt.subplot(1, 2, 1)
plt.plot(rdf_params.rdf_bin_centers, rdf_params.reference, label="Experimental RDF")
plt.plot(rdf_params.rdf_bin_centers, rdf_observable(trajectory.aux), label="Simulated RDF")
plt.xlabel("Distance")
plt.ylabel("RDF")
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(adf_params.adf_bin_centers, adf_params.reference, label="Experimental ADF")
plt.plot(adf_params.adf_bin_centers, adf_observable(trajectory.aux), label="Simulated ADF")
plt.legend()
plt.xlabel("Angle")
plt.ylabel("ADF")

## DiffTRe Fine-Tuning

We now switch from force matching to fitting ensemble-averaged structural
observables. The trainer is initialized from the best many-body force-matching
parameters and refines them so that the RDF and ADF agree more closely with the
target curves.

For a reference trajectory $\{\mathbf R_n\}_{n=1}^N$ generated under a reference potential $\tilde U$, DiffTRe estimates observables under the current model $U_\theta$ by reweighting,
$$
\langle a \rangle_\theta \approx \sum_{n=1}^N w_n\, a(\mathbf R_n),
\qquad
w_n=
\frac{\exp\left[-\beta\left(U_\theta(\mathbf R_n)-\tilde U(\mathbf R_n)\right)\right]}
{\sum_{m=1}^N \exp\left[-\beta\left(U_\theta(\mathbf R_m)-\tilde U(\mathbf R_m)\right)\right]}.
$$
With the default loss, the structural targets enter through a weighted mean-squared error,
$$
\mathcal L(\theta)
=
\gamma_{\mathrm{RDF}} \left\| \hat g_{\mathrm{RDF}} - g_{\theta}^{\mathrm{RDF}} \right\|^2
+
\gamma_{\mathrm{ADF}} \left\| \hat g_{\mathrm{ADF}} - g_{\theta}^{\mathrm{ADF}} \right\|^2,
$$
and here both weights are set to $1.0$.

The call to `add_statepoint(...)` collects all ingredients for one thermodynamic
state point: the many-body energy model, the sampling protocol, the state
variables `box` and `kT`, the instantaneous quantities, the observables, the
target RDF and ADF curves, the neighbor list, and the initial reference state.

During training, DiffTRe then alternates between three steps:

1) it generates or reuses a reference trajectory
2) it computes reweighted observable averages and the corresponding loss
3) it updates the model parameters from the resulting gradients

Trajectory reuse is controlled through the effective sample size
$$
N_{\mathrm{eff}} = \exp\left(-\sum_{n=1}^N w_n \log w_n\right),
$$
and `reweight_ratio=0.5` means that the stored trajectory is reused as long as
at least half of the snapshots remain effectively informative; otherwise a new
trajectory is generated.


In [ ]:
num_iter = 50
lr_decay = optax.polynomial_schedule(init_value=2e-4, end_value=5e-5, power=2, transition_steps=num_iter)
optimizer = optax.adam(learning_rate=lr_decay, b1=0.5, b2=0.75)

difftre_trainer = trainers.Difftre(
    manybody_fm_trainer.best_params,
    optimizer,
    reweight_ratio=0.5)
difftre_trainer.add_statepoint(
    energy_fn_template=manybody_energy_fn_template,
    simulator_template=sim_template,
    timings=timings,
    state_kwargs={"box": box, "kT": kT},
    quantities=quantities,
    observables={
        "rdf": rdf_observable,
        "adf": adf_observable
    },
    targets={
        "rdf": {
            "target": rdf_params.reference,
            "gamma": 1.0
        },
        "adf": {
            "target": adf_params.reference,
            "gamma": 1.0
        }
    },
    neighbor_fn=nbrs_init,
    reference_state=reference_state
)

This call starts the DiffTRe optimization for the `num_iter` iterations defined
above. In each step, the trainer evaluates the current model against the
structural targets, computes the loss from the reweighted RDF and ADF
observables, and updates the parameters with the optimizer defined above.

In [ ]:
difftre_trainer.train(num_iter)

In [ ]:
plt.plot(difftre_trainer.batch_losses, label="DiffTRe Train Loss")

Finally, we compare three quantities on the same axes: the experimental target
curves, the RDF and ADF obtained from the stored trajectory of the pretrained
many-body force-matching model (`trajectory.aux`), and the stored DiffTRe
prediction `difftre_trainer.predictions[0][num_iter - 1]`.

The DiffTRe curves shown here are therefore reweighted predictions from the
trainer, not observables from a newly simulated fine-tuned trajectory.

In [ ]:
plt.figure(figsize=(12, 5), layout="constrained")
plt.subplot(1, 2, 1)
plt.plot(rdf_params.rdf_bin_centers, difftre_trainer.predictions[0][num_iter - 1]["rdf"], label="DiffTRe")
plt.plot(rdf_params.rdf_bin_centers, rdf_params.reference, label="Experimental RDF")
plt.plot(rdf_params.rdf_bin_centers, rdf_observable(trajectory.aux), label="Simulated RDF")
plt.xlabel("Distance")
plt.ylabel("RDF")
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(adf_params.adf_bin_centers, difftre_trainer.predictions[0][num_iter - 1]["adf"], label="DiffTRe")
plt.plot(adf_params.adf_bin_centers, adf_params.reference, label="Experimental ADF")
plt.plot(adf_params.adf_bin_centers, adf_observable(trajectory.aux), label="Simulated ADF")
plt.legend()
plt.xlabel("Angle")
plt.ylabel("ADF")

The comparison shows whether structural fine-tuning improves agreement with the
reference observables beyond the pretrained many-body simulation. In this plot,
the DiffTRe prediction moves the RDF and ADF closer to the targets than the
baseline trajectory in several regions, which is exactly what DiffTRe is meant
to do.

## Outlook: Using MLP in LAMMPS

To use the trained potential in LAMMPS, we need to export it into the `chemtrain-deploy` format expected by the connector. This cell defines the deployment-time energy function and runs the export step that packages the model together with the graph and unit metadata needed later on the LAMMPS side.

One non-obvious part is that the model is rebuilt inside `energy_fn`. The reason is that deployment uses a graph object whose buffer sizes can differ from the training setup, in particular the maximum number of retained edges after pruning. `initialize_model(..., graph.max_edges.size)` recreates an `apply_fn` whose static graph-related shapes match the exported interface, so the compiled model is consistent with the neighbor-list capacity seen during deployment. In the same function, positions are converted back from angstroms to nanometers before evaluation, the species are fixed to a single-species encoding, and the predicted energy is converted from `kJ/mol` to `kcal/mol` to match `unit_style = "real"`.

We stop here at `model.export()`, which is the point where the deployable representation is created. To continue, see:
- [`chemtrain.deploy.exporter.Exporter`](https://chemtrain.readthedocs.io/en/latest/api/deploy/exporter.html) for the export workflow and the expected `energy_fn`
- [`chemtrain.deploy.graphs`](https://chemtrain.readthedocs.io/en/latest/api/deploy/graphs.html) for `SimpleDenseNeighborList` and graph buffering
- [`chemtrain-deploy` Getting Started](https://chemtrain.readthedocs.io/en/latest/chemtrain-deploy/getting_started.html) for loading the exported model in LAMMPS
- [`chemtrain-deploy` Installation](https://chemtrain.readthedocs.io/en/latest/chemtrain-deploy/installation.html) for setting up the connector and plugin


In [ ]:
from chemtrain.deploy import exporter, graphs

free_displacement, _ = space.free()
class ExportedModel(exporter.Exporter):

    unit_style = 'real'
    r_cutoff = r_cutoff * 10 # Units: A, kcal/mol
    graph_type = graphs.SimpleDenseNeighborList
    nbr_order = [1, 2]

    def energy_fn(self, position, species, graph):
        neighbor = graph.to_neighborlist()

        _, apply_fn = initialize_model(
            architecture, config, r_cutoff, free_displacement, graph.max_edges.size)

        species = jnp.zeros((position.shape[0],), dtype=int)

        position = position / 10.0 # Convert from A to nm
        pot = apply_fn(manybody_fm_trainer.best_params, position, species, neighbor)
        pot /= 4.184 # Convert from kJ/mol to kcal/mol

        return pot.squeeze(axis=-1)

model = ExportedModel()
model.export()
model.save(data_dir / "exported_model.ptb")